# Single final evaluation of the Jul–Aug 2018 period

This notebook opens the test period, from 2018-07-01 to 2018-08-31, exactly once, to answer two questions fixed before execution. The first is whether CatBoost (Prokhorenkova et al., 2018), trained on the history available up to 2018-05-31, keeps outside the development period the advantage over the best baseline observed in the folds. The second is whether the predicted probabilities remain calibrated under the known drift of the period, in which the median promised lead time decreases relative to the training history.

The notebook does not select models, tune hyperparameters or change the data contract. It measures and documents, following the pre-registration P-05 (Nosek et al., 2018) in `docs/REGISTRO-DE-DECISOES.md` and the plan in `docs/PLANO-NOTEBOOK-09.md`.

## Pre-registration

The table summarizes what was fixed before the test was opened. No item can be changed after the result is known.

| Item | Pre-registered definition |
|---|---|
| Model | CatBoost obtained from `build_classification_tournament` with the tournament default configuration (`iterations=300`, `learning_rate=0.05`, `depth=6`, `random_seed=42`), without class weights and with isotonic calibration `cv=3` fitted on the training data (T-05) |
| Features | Current `NUMERIC_FEATURES` and `CATEGORICAL_FEATURES`, including the as-of network and `distance_mean_km` (P-01, P-03) |
| Training | Purchases before 2018-06-01, that is, up to and including 2018-05-31; June 2018 enters no step |
| Test | Supervised orders purchased in [2018-07-01, 2018-09-01); `shipped` orders without delivery are left out (P-02) |
| Baselines | Prevalence, negative promised lead time and logistic promise + state, fitted on the same training window; the main reference is `baseline_logistic_promise_plus_state` |
| Metrics | Average precision as the primary metric (Davis and Goadrich, 2006), ROC AUC (Fawcett, 2006), Brier (Brier, 1950), ECE with 10 bins (Naeini et al., 2015), reliability curve (DeGroot and Fienberg, 1983), and precision and recall in the top 5% and 10% of risk |
| Uncertainty | Weekly block bootstrap (ISO week of purchase), 1,000 resamples, seed 42, 95% percentile CI (Künsch, 1989; Efron and Tibshirani, 1993); single-class resamples are discarded and counted |
| Reading | Paired AP difference (CatBoost minus logistic baseline): confirmed if the difference is positive and the lower CI bound is above zero; not confirmed if the difference is negative and the upper bound is below zero; inconclusive otherwise |
| Regression | Descriptive only (MAE and RMSE), with no winner declared (P-04) |
| Drift | Train × test PSI (Siddiqi, 2006) on the numeric features, highlighting `promised_lead_days` and the network rates; monthly prevalence; right-censoring sensitivity in counts and prevalence only |

## Single-use rule

The evaluation is executed by `run_final_evaluation`, the only call in the project with `allow_test=True`. The result of each version is stored in `outputs/runs/final_evaluation/<version>/` with frozen semantics: when the version directory already exists, the stored result is always loaded, even if data, configuration or package code have changed, and a warning reports the fingerprint divergence. The version is never recomputed. Each computation appends one line to the append-only ledger `LEDGER.jsonl` in the same directory, and the trained final model is saved with joblib and `metadata.json`. Consequently, running the notebook in the second language only loads the result produced in the first.

A new evaluation requires a new version (for example, F2) and is labeled post hoc in any text that cites it. Earlier versions of the project, with another target and other features, have already consulted this period; therefore, the period is not a pristine holdout. Version F1 is the first evaluation made with the current contract, with no model, feature or hyperparameter selection based on it.

## Data and populations

The data are loaded and transformed exactly as in notebook 07: supervised population, purchase and as-of network features, and a modeling table with one row per order. The training and test windows are applied inside `run_final_evaluation`; the next cell only prepares the table and reports the fixed boundaries.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

LANG = "en"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    FINAL_TEST_END,
    FINAL_TEST_START,
    FINAL_TRAIN_END,
    build_modeling_frame,
    build_order_features,
    build_supervised_orders,
    describe_run,
    read_olist_tables,
    run_final_evaluation,
)

FINAL_VERSION = "F1"
N_BOOT = 1000
SEED = 42
CACHE_ROOT = project_root / "outputs" / "runs"
MODEL_DIR = project_root / "outputs" / "models" / "catboost_final"

LABELS = {
    "pt": {
        "ideal": "calibração perfeita",
        "x": "probabilidade média prevista",
        "y": "taxa observada",
        "title": "Curva de confiabilidade (10 bins)",
        "discarded": "Reamostragens descartadas por conter uma só classe: {n} de {total}",
        "reading": "Leitura pré-registrada da diferença pareada de AP (CatBoost - baseline logística): {outcome}",
        "post_hoc": "Execução pós-hoc (versão diferente de F1): {flag}",
        "eligible": "production_eligible = {value}",
        "train": "Treino: compras anteriores a {end}",
        "test": "Teste: compras em [{start}, {end})",
        "available": "Pedidos supervisionados disponíveis: {n}",
    },
    "en": {
        "ideal": "perfect calibration",
        "x": "mean predicted probability",
        "y": "observed rate",
        "title": "Reliability curve (10 bins)",
        "discarded": "Resamples discarded for containing a single class: {n} of {total}",
        "reading": "Pre-registered reading of the paired AP difference (CatBoost - logistic baseline): {outcome}",
        "post_hoc": "Post-hoc run (version other than F1): {flag}",
        "eligible": "production_eligible = {value}",
        "train": "Train: purchases before {end}",
        "test": "Test: purchases in [{start}, {end})",
        "available": "Supervised orders available: {n}",
    },
}[LANG]

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
print(LABELS["train"].format(end=FINAL_TRAIN_END))
print(LABELS["test"].format(start=FINAL_TEST_START, end=FINAL_TEST_END))
print(LABELS["available"].format(n=len(modeling_frame)))

Train: purchases before 2018-06-01
Test: purchases in [2018-07-01, 2018-09-01)
Supervised orders available: 96470


### Insight

The modeling table contains every supervised order, and the split into train and test is made only in the next step, by package functions that verify the boundaries. The effective window sizes appear in the tables of the next section, and this text does not reproduce them so that it does not become outdated relative to the execution.

## Single execution and populations

The next cell calls `run_final_evaluation` with `allow_test=True` and version F1. On the first execution, the model is trained, the test is evaluated, the tables are stored, the model is saved and a line is appended to the ledger. On any later execution, including in the other language, the frozen result is loaded. The provenance printed by `describe_run` states which of the two cases occurred, and the tables describe the populations, the right-censoring sensitivity (counts and prevalence only, per P-02) and the ledger.

In [2]:
final, final_meta = run_final_evaluation(
    modeling_frame,
    allow_test=True,
    version=FINAL_VERSION,
    cache_root=CACHE_ROOT,
    model_dir=MODEL_DIR,
    n_boot=N_BOOT,
    seed=SEED,
    orders=tables["orders"],
)
print(describe_run(final_meta, language=LANG))
print(LABELS["post_hoc"].format(flag=final_meta["post_hoc"]))

display(final["provenance"])
display(final["populations"])
if "censoring" in final:
    display(final["censoring"])
ledger_path = CACHE_ROOT / "final_evaluation" / "LEDGER.jsonl"
if ledger_path.exists():
    display(pd.read_json(ledger_path, lines=True))

Results loaded from outputs/runs/final_evaluation/F1 (generated on 2026-11-14 10:53, 1.3 min of training; fingerprint ddc0ef97)
Post-hoc run (version other than F1): False


TEMP\ipykernel_20972\3819203259.py:1: UserWarning: Final evaluation F1 already exists and is frozen: loading PROJECT_ROOT\outputs\runs\final_evaluation\F1 (stored fingerprint ddc0ef97) although the current fingerprint is 9ef9dda3 (configuration, data or code changed). Nothing was recomputed; use a new version for a post-hoc run.
  final, final_meta = run_final_evaluation(


,version,post_hoc,created_at,fingerprint,train_orders,test_orders,train_end_exclusive,test_start,test_end_exclusive,n_boot,seed,model_path
0,F1,False,2026-11-14T10:53:16,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,2018-06-01,2018-07-01,2018-09-01,1000,42,PROJECT_ROOT...


,split,window_start,window_end_exclusive,orders,breaches,prevalence,first_purchase,last_purchase,weeks
0,train,None,2018-06-01,77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,NaN
1,test,2018-07-01,2018-09-01,12507,601,0.048053,2018-07-01 00:25:07,2018-08-29 15:00:37,10.0


,scenario,supervised_orders,breaches,censored_orders,prevalence
0,exclude,12507,601,107,0.048053
1,include_as_breach,12614,708,107,0.056128


,version,timestamp,fingerprint,train_orders,test_orders,post_hoc,n_boot,seed,gate_passed
0,F1,2026-11-14 10:53:17,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,False,1000,42,NaN
1,F2,2026-11-14 12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,88097,6351,True,1000,42,"C,E"


### Insight

The provenance line states whether the results were computed now or loaded, and the ledger holds one line per computation. The training and test populations, the breach prevalence and the censoring scenarios are in the tables above. The `include_as_breach` scenario is a descriptive upper bound and feeds no model metric, so the evaluated population is that of delivered orders with valid dates.

## Train versus test drift

The PSI compares each numeric model feature between the training and test windows, with quantiles defined on the training data. The promised lead time (`promised_lead_days`) and the historical network rates are highlighted. The monthly breach prevalence is presented for the training and test months, without June 2018. The bar chart on a logarithmic scale ranks every feature by PSI, with the calendar features in grey and the highlighted ones in their own colors.

In [3]:
psi = final["psi"]
display(psi.head(10))
display(psi[psi["highlight"] != "other"])
display(final["monthly_prevalence"])

from ecom_delivery_sla_viz import plot_psi

calendar_features = ["purchase_month", "days_to_black_friday", "days_to_christmas"]
plot_psi(psi, neutral_features=calendar_features, highlight_col="highlight", language=LANG).show()

,feature,psi,highlight
0,days_to_black_friday,10.913776,other
1,days_to_christmas,10.741474,other
2,purchase_month,10.687575,other
3,dest_hist_breach_rate,6.303497,network_rate
4,dest_hist_median_lead_days,4.301667,other
5,dest_in_degree,1.395556,other
6,seller_hist_breach_rate,1.317856,network_rate
7,edge_hist_breach_rate,1.145810,network_rate
8,promise_slack,0.957536,other
9,promised_lead_days,0.767983,promise


,feature,psi,highlight
3,dest_hist_breach_rate,6.303497,network_rate
6,seller_hist_breach_rate,1.317856,network_rate
7,edge_hist_breach_rate,1.145810,network_rate
9,promised_lead_days,0.767983,promise
20,seller_hist_dispatch_late_rate,0.023378,network_rate


,purchase_month,split,orders,breaches,prevalence
0,2016-09,train,1,1,1.000000
1,2016-10,train,265,2,0.007547
2,2016-12,train,1,0,0.000000
3,2017-01,train,750,22,0.029333
4,2017-02,train,1653,49,0.029643
5,2017-03,train,2546,116,0.045562
6,2017-04,train,2303,151,0.065567
7,2017-05,train,3545,106,0.029901
8,2017-06,train,3135,95,0.030303
9,2017-07,train,3872,108,0.027893


### Insight

The PSI is a descriptive diagnostic and defines no decision thresholds in this notebook. The per-feature values, sorted from largest to smallest, and the monthly prevalence trend are in the tables above. Any performance drop should be read in light of these distribution shifts, without causal attribution, because the PSI measures marginal shift and not the effect on the model. The calendar features appear in grey in the chart because their PSI is high by construction.

## Classification: metrics with intervals and paired difference

The first table reports, for CatBoost and the three baselines, the point estimate on the full sample and the 95% CI of the weekly block bootstrap, with every model evaluated on the same resamples. The second table reports the paired difference in average precision between CatBoost and `baseline_logistic_promise_plus_state`, with its 95% CI, and the reading follows the pre-registered rule. Brier and ECE are not defined for the `negative_promised_lead_days` score, which is not a probability. An interval chart complements the tables, with the average precision estimate and the 95% CI of each model and, beside it, the paired difference relative to zero.

In [4]:
metrics = final["classification_metrics"]
display(metrics[["model", "metric", "estimate", "ci_low", "ci_high"]])
print(LABELS["discarded"].format(n=int(metrics["n_discarded"].iloc[0]), total=int(metrics["n_boot"].iloc[0])))

paired = final["paired_difference"]
display(paired[["model", "reference", "estimate_model", "estimate_reference", "difference", "ci_low", "ci_high"]])
outcome = paired.loc[0, "outcome"] if LANG == "pt" else paired.loc[0, "outcome_en"]
print(LABELS["reading"].format(outcome=outcome))

from ecom_delivery_sla_viz import plot_metric_intervals

plot_metric_intervals(metrics, metric="average_precision", paired=paired, language=LANG).show()

,model,metric,estimate,ci_low,ci_high
0,CatBoost,average_precision,0.081983,0.055835,0.133698
1,CatBoost,roc_auc,0.610157,0.561247,0.661876
2,CatBoost,brier,0.050476,0.031599,0.068670
3,CatBoost,ece_10bins,0.038628,0.033804,0.054392
4,CatBoost,precision_at_5pct,0.108626,0.079020,0.152373
5,CatBoost,recall_at_5pct,0.113145,0.070758,0.225214
6,CatBoost,precision_at_10pct,0.084732,0.058771,0.110333
7,CatBoost,recall_at_10pct,0.176373,0.120073,0.305240
8,baseline_prevalence,average_precision,0.048053,0.027216,0.068543
9,baseline_prevalence,roc_auc,0.500000,0.500000,0.500000


Resamples discarded for containing a single class: 0 of 1000


,model,reference,estimate_model,estimate_reference,difference,ci_low,ci_high
0,CatBoost,baseline_logistic_promise_plus_state,0.081983,0.08425,-0.002267,-0.032492,0.063988


Pre-registered reading of the paired AP difference (CatBoost - logistic baseline): inconclusive


### Insight

The pre-registered reading follows exclusively from the sign of the point difference and the position of the 95% CI relative to zero. The values are in the tables above, and this text does not reproduce them. The number of discarded resamples, printed after the first table, indicates how many week resamples did not contain both classes. Wide intervals reflect the limited number of weeks in the test period and the correlation within each week, and should not be read as evidence of no effect. In the chart, the position of the paired-difference interval relative to the zero line corresponds to the pre-registered reading.

## Calibration and prioritization

The reliability curve compares, in ten probability bins, the mean predicted probability with the observed breach rate, for the models that produce probabilities. The precision and recall values in the top 5% and 10% of risk, with their intervals, summarize the use of the score for prioritization.

In [5]:
reliability = final["reliability"]
display(reliability)

figure = go.Figure()
for model, part in reliability.groupby("model", sort=False):
    figure.add_trace(
        go.Scatter(x=part["mean_predicted"], y=part["observed_rate"], mode="lines+markers", name=model)
    )
figure.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line={"dash": "dash"}, name=LABELS["ideal"])
)
figure.update_layout(title=LABELS["title"], xaxis_title=LABELS["x"], yaxis_title=LABELS["y"])
figure.show()

display(final["top_k"])

,model,bin,bin_lower,bin_upper,n,mean_predicted,observed_rate
0,CatBoost,1,0.0,0.1,9080,0.048305,0.043833
1,CatBoost,2,0.1,0.2,2227,0.140257,0.044903
2,CatBoost,3,0.2,0.3,881,0.243339,0.063564
3,CatBoost,4,0.3,0.4,244,0.338651,0.106557
4,CatBoost,5,0.4,0.5,57,0.437436,0.245614
5,CatBoost,6,0.5,0.6,9,0.548315,0.555556
6,CatBoost,7,0.6,0.7,5,0.632761,0.200000
7,CatBoost,8,0.7,0.8,3,0.728085,0.333333
8,CatBoost,9,0.8,0.9,1,0.800651,0.000000
9,CatBoost,10,0.9,1.0,0,NaN,NaN


,model,top_fraction,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high
0,CatBoost,0.05,0.108626,0.079020,0.152373,0.113145,0.070758,0.225214
1,CatBoost,0.10,0.084732,0.058771,0.110333,0.176373,0.120073,0.305240
2,baseline_prevalence,0.05,0.047923,0.009601,0.093909,0.049917,0.010360,0.100013
3,baseline_prevalence,0.10,0.045564,0.013006,0.096087,0.094842,0.027850,0.217651
4,baseline_negative_promised_lead_days,0.05,0.190096,0.064200,0.283750,0.198003,0.105552,0.257891
5,baseline_negative_promised_lead_days,0.10,0.167866,0.058809,0.239437,0.349418,0.195506,0.418261
6,baseline_logistic_promise_plus_state,0.05,0.115016,0.078062,0.160643,0.119800,0.096207,0.154398
7,baseline_logistic_promise_plus_state,0.10,0.082334,0.056623,0.108987,0.171381,0.140491,0.226266


### Insight

The closeness of the points to the diagonal indicates calibration, and the number of orders per bin (`n`) shows where the curve is stable. The values are in the tables and in the figure above. The isotonic calibration was fitted on the training data only, so deviations on the test reflect the distribution shift of the period and not later adjustment.

## Descriptive regression

The MAE and RMSE of the delay, in days, are reported on the test for the state baseline (`state_median_actual_minus_promise`), for the other baselines and for the default CatBoost regressor, all trained on the same window. Consistent with P-04, the regression is only descriptive and no winner is declared.

In [6]:
display(final["regression"].sort_values("mae"))

,model,mae,rmse,n_test
3,CatBoost,3.885680,5.331360,12507
2,baseline_state_median_actual_minus_promise,4.514864,5.723027,12507
1,baseline_median_actual_minus_promise,4.927942,6.079787,12507
0,baseline_train_mean,6.392634,8.215881,12507


### Insight

The metrics are in the table above, sorted by MAE only to ease reading. The ordering is not model selection, and the delay regression remains a descriptive analysis, with the state baseline as the reference. No confidence interval is computed for the regression, so small differences should not be interpreted.

## Reading by the pre-registered criterion and limitations

The last cell repeats the pre-registered reading of the paired AP difference and fixes the `production_eligible` field. No other metric takes part in the reading decision.

In [7]:
reading = final["paired_difference"].loc[0]
display(reading[["difference", "ci_low", "ci_high", "outcome", "outcome_en"]].to_frame().T)
print(LABELS["reading"].format(outcome=reading["outcome"] if LANG == "pt" else reading["outcome_en"]))
production_eligible = False
print(LABELS["eligible"].format(value=production_eligible))

,difference,ci_low,ci_high,outcome,outcome_en
0,-0.002267,-0.032492,0.063988,inconclusivo,inconclusive


Pre-registered reading of the paired AP difference (CatBoost - logistic baseline): inconclusive
production_eligible = False


### Insight

The reading (confirmed, not confirmed or inconclusive) follows the rule fixed in P-05 and is not reinterpreted after the result. Four limitations must accompany any citation of the result. First, right censoring: `shipped` orders without delivery in July and August 2018 were excluded (P-02), so the test population is conditioned on observed delivery, and the `include_as_breach` scenario serves only as a descriptive upper bound. Second, the drift of the promised lead time: the distribution of the promise changes between training and test, and the advantage over the baseline may reflect that change. Third, the network history carries a label-delay bias: the outcome of an order enters the history only when delivery is known, so late deliveries reach the history with a lag and recent rates tend to understate delays. Fourth, the period was already consulted by earlier versions of the project, with another target and other features, and is therefore not a pristine holdout.

Any re-evaluation with another configuration receives another version (F2 and later) and is post hoc. The temporal, network and feature contract gates do not authorize operational use on the basis of a single two-month window of an exploratory dataset; therefore `production_eligible` remains `false`, following decision 21 of the registry.

## References

Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.

Davis, J. and Goadrich, M. (2006). The relationship between precision-recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233–240. https://doi.org/10.1145/1143844.1143874

DeGroot, M. H. and Fienberg, S. E. (1983). The comparison and evaluation of forecasters. *The Statistician*, 32(1/2), 12–22. https://doi.org/10.2307/2987588

Efron, B. and Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.

Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *Annals of Statistics*, 17(3), 1217–1241. https://doi.org/10.1214/aos/1176347265

Naeini, M. P., Cooper, G. F., and Hauskrecht, M. (2015). Obtaining well calibrated probabilities using Bayesian binning. *Proceedings of the 29th AAAI Conference on Artificial Intelligence*, 2901–2907. https://doi.org/10.1609/aaai.v29i1.9602

Nosek, B. A., Ebersole, C. R., DeHaven, A. C., and Mellor, D. T. (2018). The preregistration revolution. *Proceedings of the National Academy of Sciences*, 115(11), 2600–2606. https://doi.org/10.1073/pnas.1708274114

Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., and Gulin, A. (2018). CatBoost: unbiased boosting with categorical features. *Advances in Neural Information Processing Systems 31*.

Siddiqi, N. (2006). *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring*. Wiley.